# Robust Conversational Chains with LangChain and OpenAI

1. **ConversationalRetrievalChain** - Managing conversational context
2. **Memory Management** - Techniques for maintaining conversation history
3. **Security Guardrails** - Filters for safe responses
4. **Re-ranking** - Improving retrieval quality

---

## 📦 Installation and Imports

First, let's install the necessary dependencies and import the libraries.

In [ ]:
!pip install langchain openai langchain-community chromadb tiktoken

In [ ]:
# Necessary imports
import os
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain.chat_models import ChatOpenAI
import warnings
from dotenv import load_dotenv
warnings.filterwarnings('ignore')


## 🔧 Initial Setup

Let's configure the OpenAI API key and initialize the basic components.

In [ ]:
# Configuration of the OpenAI API key
# You can get your key at: https://platform.openai.com/account/api-keys


# Load the .env file
load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print(f"API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## 📚 1. Data Preparation

Let's create some sample documents to demonstrate the concepts.

In [ ]:
# Creation of sample documents on AI and Machine Learning
example_documents = [
    """Artificial Intelligence (AI) is a field of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes learning, reasoning, perception, and decision-making.""",

    """Machine Learning is a subfield of AI that enables computers to learn and improve automatically through experience without being explicitly programmed. ML algorithms identify patterns in data and make predictions.""",

    """Deep Learning is a machine learning technique based on artificial neural networks with multiple layers. It is particularly effective for tasks such as image recognition, natural language processing, and speech recognition.""",

    """RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It allows language models to access external knowledge to generate more accurate and up-to-date responses.""",

    """LangChain is a framework for developing applications with language models. It facilitates the creation of complex chains, memory management, and integration with various data sources.""",

    """Google Gemini is a multimodal language model developed by Google, capable of processing text, images, and code. It offers advanced reasoning and contextual understanding capabilities."""
]

# Conversion to Document objects
docs = [Document(page_content=doc) for doc in example_documents]

print(f"✅ Created {len(docs)} example documents")

## 🔍 2. Creating the Vector Store with OpenAI Embeddings

Let's create a vector store using Chroma with embeddings from OpenAI.

In [ ]:
# Initialize OpenAI embeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=api_key
)

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding= embeddings,
    persist_directory="./chroma_db_openai")

print(f"Number of indexed documents: {vectorstore._collection.count()}")

## 🧠 3. Memory Management

Implementing **ConversationBufferWindowMemory** to maintain conversation context.

In [ ]:
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)


print("✅ Memory configured!")
print(memory.k)

## 🔗 4. ConversationalRetrievalChain with OpenAI

Creating the conversational chain that combines document retrieval with the OpenAI model.

In [ ]:
# Initialize OpenAI model

llm = ChatOpenAI(
    model="gpt-5-nano",
    openai_api_key=api_key,
)


# Create the ConversationalRetrievalChain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=True,
    verbose=True
)



## 💬 5. Testing the Conversation

Let's test our conversational chain with some questions.

In [ ]:
def ask_question(question):
    """Helper function to ask questions to the conversational chain"""

    print(f"\n Question: {question}")
    print("-" * 50)

    try:
        result = qa_chain({"question": question})

        print(f"🤖 Answer: {result['answer']}")
        print(f"\n📚 Documents used: {len(result['source_documents'])}")

        return result
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

# First question
result1 = ask_question("What is Artificial Intelligence?")

In [ ]:
result2 = ask_question("How is it related to Machine Learning")

In [ ]:
result3 = ask_question("And what is Google Gemini as you mentioned?")

## 🛡️ 6. Security Guardrails

Implementation of security filters to prevent inappropriate responses.

In [ ]:
import re

class SecurityGuardrails:
    def __init__(self):
        self.prohibited_words = [
            'password', 'cpf', 'rg', 'credit card',
            'personal data', 'confidential information', 'api key',
            'api key', 'access token'
        ]

        self.pii_patterns = [
            r'\d{3}\.\d{3}\.\d{3}-\d{2}',        # CPF pattern (999.999.999-99)
            r'\d{4}\s?\d{4}\s?\d{4}\s?\d{4}',    # Credit card pattern (16 digits)
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',  # Email pattern
            r'AIza[0-9A-Za-z\-_]{35}'            # Google API Key pattern
        ]

    def check_question(self, question):
        """Checks if the question contains inappropriate content"""
        question_lower = question.lower()

        # Check prohibited words
        for word in self.prohibited_words:
            if word in question_lower:
                return False, f"Question contains inappropriate term: {word}"

        # Check PII patterns
        for pattern in self.pii_patterns:
            if re.search(pattern, question):
                return False, "Question contains personal information"

        return True, "Question approved"

    def check_answer(self, answer):
        """Checks if the answer is appropriate"""
        answer_lower = answer.lower()

        # Check if the answer is within scope
        scope_terms = ['ai', 'artificial intelligence', 'machine learning',
                         'deep learning', 'rag', 'langchain', 'gemini', 'google']  # Allowed topics

        has_scope_term = any(term in answer_lower for term in scope_terms)  # Checks if there's at least one relevant term

        if not has_scope_term and len(answer) > 50:  # If it's out of scope and still long
            return False, "Answer out of application scope"  # Block it

        # Check if it doesn't contain sensitive information
        for pattern in self.pii_patterns:                 # Iterate through each PII regex
            if re.search(pattern, answer):             # If sensitive data is found
                return False, "Answer contains sensitive information"  # Block it

        return True, "Answer approved"  # Return success if everything is OK

# Initialization of the guardrails
guardrails = SecurityGuardrails()  # Create the instance of the guardrails
print("✅ Security guardrails configured!")


In [ ]:
def safe_question(question):
    """Function that applies guardrails before processing the question"""
    approved, message = guardrails.check_question(question)

    if not approved:
        print(f"🚫 Question rejected: {message}")
        return None

    try:
        # Process question
        result = qa_chain({"question": question})

        # Check answer
        approved_resp, message_resp = guardrails.check_answer(
            result['answer']
        )

        if not approved_resp:
            print(f"🚫 Answer rejected: {message_resp}")
            return None

        print(f"✅ {message}")
        print(f"✅ {message_resp}")
        print(f"\n🤖 Answer: {result['answer']}")

        return result

    except Exception as e:
        print(f"❌ Error processing question: {str(e)}")
        return None

# Test with appropriate question
print("=== Test with appropriate question ===")
safe_question("Explain Deep Learning")


In [ ]:
# Test with inappropriate question
print("\n=== Test with inappropriate question ===")
safe_question("What is your API key?")

## 🔄 7. Improved Re-ranking

Implementation of re-ranking to improve the quality of document retrieval.

In [ ]:
import numpy as np  # Import NumPy, although it's not used directly here (could be removed)
from sklearn.metrics.pairwise import cosine_similarity  # Function to calculate cosine similarity between vectors

class RerankOpenAI:  # Class responsible for reordering (re-ranking) documents based on OpenAI embeddings
    def __init__(self, embeddings_model):  # Constructor that receives an embeddings model
        self.embeddings_model = embeddings_model  # Stores the provided embedder
        self.name = "Re-ranking with OpenAI Embeddings"  # Descriptive name for identification

    def rerank(self, query, documents, top_k=3):  # Main re-ranking method; returns top_k most relevant docs
        """Re-ranking based on semantic similarity using OpenAI embeddings"""  # Explanatory docstring
        try:  # Try to execute the main flow (might fail, so there's a fallback)
            # Generate query embedding
            query_embedding = self.embeddings_model.embed_query(query)  # Creates vector of the question using the embeddings model

            # Generate document embeddings
            doc_texts = [doc.page_content if hasattr(doc, 'page_content') else str(doc) for doc in documents]  # Extracts text from each Document
            doc_embeddings = self.embeddings_model.embed_documents(doc_texts)  # Converts texts to vectors

            # Calculate similarities
            similarities = cosine_similarity([query_embedding], doc_embeddings)[0]  # Calculates cosine similarity between query and each doc

            # Create list of documents with scores
            scored_docs = list(zip(similarities, documents))  # Combines score and doc in tuples

            # Sort by similarity (highest first)
            scored_docs.sort(key=lambda x: x[0], reverse=True)  # Sorts by descending score

            # Return top_k documents
            return [doc for _, doc in scored_docs[:top_k]]  # Returns only the most relevant documents

        except Exception as e:  # Catches exceptions (e.g.: API failure)
            print(f"Error in re-ranking: {e}")  # Displays error message
            # Fallback to simple re-ranking
            return self._simple_rerank(query, documents, top_k)  # Uses secondary method in case of error

    def _simple_rerank(self, query, documents, top_k):  # Fallback method for re-ranking by keyword intersection
        """Fallback: simple re-ranking based on keywords"""  # Docstring
        query_words = set(query.lower().split())  # Splits the query into words (lowercase) for comparison

        scored_docs = []  # List for (score, doc)
        for doc in documents:  # Iterates over documents
            doc_text = doc.page_content if hasattr(doc, 'page_content') else str(doc)  # Gets text from the doc
            doc_words = set(doc_text.lower().split())  # Converts text to set of words
            score = len(query_words.intersection(doc_words)) / len(query_words) if query_words else 0  # Percentage of common words
            scored_docs.append((score, doc))  # Adds (score, doc) tuple to the list

        scored_docs.sort(key=lambda x: x[0], reverse=True)  # Sorts by intersection score
        return [doc for _, doc in scored_docs[:top_k]]  # Returns top_k docs after sorting

# Initialization of the re-ranker with OpenAI
reranker = RerankOpenAI(embeddings)  # Creates instance passing the previously configured OpenAI embedder
print("✅ Re-ranker with OpenAI configured!")  # Success message when creating the re-ranker


In [ ]:
def search_with_rerank(query, k=5, top_k=3):                                      # Defines function that searches and re-ranks documents
    """Search documents with re-ranking using OpenAI"""                         # Docstring explaining the function
    print(f"🔍 Searching documents for: '{query}'")                            # Shows the query in the console

    try:                                                                        # Starts try/except block to capture errors
        # Initial search (more documents)                                       # Comment: rough search step
        initial_docs = vectorstore.similarity_search(query, k=k)               # Retrieves k most similar documents to the query
        print(f"📄 Documents found in initial search: {len(initial_docs)}")  # Displays number of returned docs

        # Re-ranking                                                            # Comment: semantic re-ranking step
        reranked_docs = reranker.rerank(query, initial_docs, top_k=top_k)      # Reorders docs via OpenAI embeddings and keeps top_k
        print(f"🎯 Documents after re-ranking: {len(reranked_docs)}")            # Shows how many docs remained after re-rank

        # Show results                                                    # Comment: loop to display snippets of the docs
        print("\n📊 Documents selected after re-ranking:")                  # Header for the final docs list
        for i, doc in enumerate(reranked_docs, 1):                              # Iterates over re-ranked docs numbering from 1
            content = doc.page_content if hasattr(doc, 'page_content') else str(doc)  # Ensures text even if not Document
            print(f"{i}. {content[:100]}...")                                   # Displays the first 100 characters of each doc

        return reranked_docs                                                    # Returns the final list of documents

    except Exception as e:                                                      # Catches possible exceptions
        print(f"❌ Error in search: {str(e)}")                                     # Shows error message in the console
        return []                                                               # Returns empty list in case of failure

# Test of re-ranking                                                           # Comment: test call of the function
docs_result = search_with_rerank("machine learning algorithms openai")         # Executes the function with a sample query


## 🎓 10. Conclusion

In this notebook, we explored the main concepts of **Robust Conversational Chains** using **OpenAI**:

✅ **ConversationalRetrievalChain**: We implemented a chain that maintains conversational context with OpenAI
✅ **Memory Management**: We configured memory to keep interaction history
✅ **Security Guardrails**: We created specific filters to protect sensitive information
✅ **Re-ranking with OpenAI**: We implemented re-ranking using OpenAI embeddings

### 🚀 Advantages of OpenAI

- **Multimodal**: Ability to process text, images, and code
- **Cost-effective**: Competitive pricing compared to other models
- **Google Integration**: Easy integration with other Google services
- **Performance**: Excellent quality of responses and reasoning
